# Pakalorie YOLOv8 Classification Training

Colab workflow for CDX-006. This keeps Gemini as the live app path and trains an offline YOLOv8 classifier for the FYP model deliverable.

In [ ]:
!nvidia-smi || true
!pip install -q ultralytics pillow pandas matplotlib seaborn scikit-learn onnx kaggle

In [ ]:
from pathlib import Path
ROOT = Path('/content/pakalorie-ml')
RAW = ROOT / 'raw'
PREPARED = ROOT / 'food-cls-224'
REPORTS = ROOT / 'reports'
RAW.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

## Download Kaggle Datasets

The current Kaggle CLI can download public datasets without a local token in this environment. If Kaggle asks for auth, upload `kaggle.json` to `/root/.kaggle/kaggle.json` and rerun this cell.

In [ ]:
!kaggle datasets download izbaiman/food-images -p {RAW / 'izbaiman-food-images'} --unzip
!kaggle datasets download useractivated/dataset -p {RAW / 'useractivated-dataset'} --unzip

## Bring Repo Scripts Into Colab

Upload or clone this repo, then set `REPO` to the repo root. In this notebook, the expected path is `/content/Pakalorie_FYP`.

In [ ]:
REPO = Path('/content/Pakalorie_FYP')
assert (REPO / 'ml/scripts/prepare_dataset.py').exists(), 'Clone or upload the repo to /content/Pakalorie_FYP first.'

In [ ]:
!python {REPO / 'ml/scripts/prepare_dataset.py'} \
  --source izbaiman/food-images={RAW / 'izbaiman-food-images/dataset'} \
  --source useractivated/dataset={RAW / 'useractivated-dataset/dataset/dataset'} \
  --output {PREPARED} \
  --audit-json {REPORTS / 'dataset_audit.json'} \
  --audit-markdown {REPORTS / 'dataset_audit.md'} \
  --class-counts-csv {REPORTS / 'class_counts.csv'} \
  --imgsz 224 \
  --val-ratio 0.2 \
  --seed 42

In [ ]:
!python {REPO / 'ml/scripts/train.py'} --data {PREPARED} --model yolov8n-cls.pt --epochs 50 --imgsz 224 --batch 64 --device 0 --project {ROOT / 'runs'} --name yolov8n_cls

In [ ]:
!python {REPO / 'ml/scripts/evaluate.py'} --weights {ROOT / 'runs/yolov8n_cls/weights/best.pt'} --data {PREPARED} --device 0 --project {ROOT / 'runs'} --name yolov8n_eval --json-out {REPORTS / 'eval_metrics.json'}

After the run, copy `best.pt`, exported `best.onnx`, `eval_metrics.json`, and confusion matrix output into the agreed artifact location, then update `ml/MODELCARD.md`.